# Crisis Negotiator — GRPO Training (Kaggle 2×T4)

Trains **Qwen2.5-3B-Instruct** with GRPO on the Crisis Negotiator RL environment.

| | |
|---|---|
| **Hardware** | Kaggle 2×T4 (16 GB each) |
| **Precision** | fp16 (T4 does not support bf16) |
| **Runtime** | ~30–45 min for 64 prompts × 4 generations |
| **Output** | `crisis-negotiator-trained/` LoRA adapter + `crisis_training_log.json` + `reward_curve_training.png` |

**Settings → Accelerator → GPU T4 ×2** before running.

In [1]:
# Cell 1 — Install dependencies
!pip install -q "openenv-core>=0.2.3" "trl>=0.14" peft accelerate datasets sentence-transformers matplotlib
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.6/174.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 18.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 4.6 MB/s eta 0:00:00
Dependencies installed.


In [7]:
import os, pathlib, subprocess, shutil                                                                                                                                                                     
                                                                                                                                                                                                             
REPO = 'https://github.com/Dinesh052/Test.git'                                                                                                                                                             
BRANCH = 'kaggle_version'                                                                                                                                                                                  
                                                                                                                                                                                                         
if not pathlib.Path('server').exists():                                                                                                                                                                    
  subprocess.run(['git', 'clone', '-b', BRANCH, REPO, '_repo'], check=True)                                                                                                                              
  for item in pathlib.Path('_repo').iterdir():                                                                                                                                                           
      dest = pathlib.Path(item.name)                                                                                                                                                                     
      if not dest.exists():                                                                                                                                                                              
          item.rename(dest)                                                                                                                                                                              
  shutil.rmtree('_repo', ignore_errors=True)                                                                                                                                                             
  print(f'Cloned {BRANCH} branch.')                                                                                                                                                                      
else:                                                                                                                                                                                                      
  print('Already inside repo.')                                                                                                                                                                          
                                                                                                                                                                                                         
print('Files:', sorted(f for f in os.listdir('.') if not f.startswith('.'))[:14])

Cloning into '_repo'...


Cloned kaggle_version branch.
Files: ['BLOG.md', 'BLOG_OLD.md', 'Dockerfile', 'README.md', 'VIDEO_SCRIPT.md', '__init__.py', '_sanity_check.py', 'baselines_run.log', 'batch_run.py', 'client.py', 'crisis_training_log.json', 'difficulty_gradient.png', 'eval.py', 'eval_baselines.py']


In [4]:
# Cell 3 — Verify GPU setup
import torch
n = torch.cuda.device_count()
for i in range(n):
    props = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {props.name} | {props.total_memory / 1024**3:.1f} GB | bf16={torch.cuda.is_bf16_supported()}')
print(f'\nTotal GPUs: {n}')
assert n >= 1, 'No GPU found! Enable GPU in Settings → Accelerator.'

GPU 0: Tesla T4 | 14.6 GB | bf16=True
GPU 1: Tesla T4 | 14.6 GB | bf16=True

Total GPUs: 2


In [8]:
# Cell 4 — Smoke-test the environment
import sys
sys.path.insert(0, '.')

from server.environment import CrisisNegotiatorEnvironment
from models import NegotiatorAction

env = CrisisNegotiatorEnvironment()
obs = env.reset(task_id='easy_domestic_desperate', seed=42)
print(f'Reset OK  | phase={obs.phase}  demands={[d["text"] for d in obs.stated_demands]}')

step_obs = env.step(NegotiatorAction(
    action_type='emotional_label',
    content="It sounds like you're feeling completely overwhelmed right now.",
    reasoning='open with empathy',
    target='hostage_taker',
))
print(f'Step OK   | reward={step_obs.reward:.4f}  phase={step_obs.phase}  done={step_obs.done}')
assert isinstance(step_obs.reward, float)
print('\nEnvironment smoke-test PASSED ✓')

Reset OK  | phase=opening  demands=['I want to talk to my kids', "Promise I won't be arrested"]


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Step OK   | reward=0.2500  phase=opening  done=False

Environment smoke-test PASSED ✓


In [12]:
# Cell 5 — Patch train_local.py for Kaggle T4 GPUs
# T4 doesn't support bf16 → use fp16
# Reduce num_generations 8→4 for speed (still enough for GRPO advantage)
# Remove deepcopy in reward fn for ~2× speedup in scoring

import pathlib

src = pathlib.Path('train_local.py').read_text()

# 1. num_generations 8 → 4
src = src.replace(
    'num_generations: int = 8             # GRPO group size',
    'num_generations: int = 4             # GRPO group size (4 for Kaggle T4 speed)',
)

# 2. bf16=True → fp16=True in GRPOConfig
src = src.replace(
    'bf16=True,',
    'bf16=False,\n        fp16=True,',
)

# 3. torch_dtype bfloat16 → float16 in model loading
src = src.replace(
    'torch_dtype=torch.bfloat16,',
    'torch_dtype=torch.float16,',
)

# 4. Replace deepcopy env scoring with heuristic-only (major speedup)
# The deepcopy block starts at "env_copy = copy.deepcopy" and ends before "# Blend:"
src = src.replace(
    '''    env_step_reward = 0.0
    outcome_bonus = 0.0
    try:
        env_copy = copy.deepcopy(st["env"])
        stepped_obs = env_copy.step(action)
        env_step_reward = float(getattr(stepped_obs, "reward", 0.0))
        # Outcome bonus: large signal when episode terminates with surrender/release
        if getattr(stepped_obs, "done", False):
            msg = getattr(stepped_obs, "message", "")
            if any(kw in (msg or "").lower() for kw in [
                "voluntary_surrender", "hostage_released", "surrender", "released",
            ]):
                outcome_bonus = 0.40  # strong signal to learn surrender-inducing sequences
            elif any(kw in (msg or "").lower() for kw in [
                "harm_event", "tactical_intervention", "supervisor_termination",
            ]):
                outcome_bonus = -0.30  # avoid catastrophic outcomes
        bd["env_step"] = round(env_step_reward, 4)
        bd["outcome_bonus"] = round(outcome_bonus, 4)
        del env_copy
    except Exception as e:
        bd["env_step"] = 0.0
        bd["outcome_bonus"] = 0.0

    # Blend: 60% heuristic (shape), 40% real env reward (signal) + outcome bonus
    blended = 0.60 * score + 0.40 * env_step_reward + outcome_bonus
    blended = max(-0.5, min(1.0, blended))''',
    '''    # Kaggle: skip deepcopy env scoring for speed — use heuristic score directly
    bd["env_step"] = 0.0
    bd["outcome_bonus"] = 0.0
    blended = score  # pure heuristic — still rich signal from phase_align + demand_ack + diversity''',
)

pathlib.Path('train_local.py').write_text(src)
print('Patched train_local.py for Kaggle T4:')
print('  ✓ num_generations: 8 → 4')
print('  ✓ bf16 → fp16')
print('  ✓ deepcopy env scoring → heuristic-only (major speedup)')

Patched train_local.py for Kaggle T4:
  ✓ num_generations: 8 → 4
  ✓ bf16 → fp16
  ✓ deepcopy env scoring → heuristic-only (major speedup)


In [13]:
!grep "torch_dtype\|bf16\|fp16\|num_generations" train_local.py

Designed for RTX 4090 16GB (Laptop) with Qwen2.5-3B-Instruct in bf16 + LoRA.
Model choice: 3B bf16 fits in ~6 GB VRAM; avoids bitsandbytes nf4 dequant
    num_generations: int = 4             # GRPO group size (4 for Kaggle T4 speed) — need >=4 for meaningful advantage variance
    """Load model with Unsloth if available, otherwise plain transformers bf16 + LoRA.
    bf16 (no 4-bit quant) is the production path: bitsandbytes nf4 dequant
    caused numerical garbage in GRPO sample loops on 3B. 3B bf16 = ~6 GB
    print(f"[model] Loading {model_name} via transformers (bf16, no quant)...")
        torch_dtype=torch.float16,
    # GRPO sampling (numerical issue with dequant in sample loop). 3B in bf16
        num_generations=CFG.num_generations,
        bf16=False,
        fp16=True,


In [16]:
import pathlib                                                                                                                                                                                             
src = pathlib.Path('train_local.py').read_text()                                                                                                                                                           
src = src.replace('max_prompt_length=1024,', '# max_prompt_length removed — not supported in this TRL version')                                                                                            
pathlib.Path('train_local.py').write_text(src)                                                                                                                                                             
print('Patched out max_prompt_length ✓')

Patched out max_prompt_length ✓


In [18]:
import pathlib                                                                                                                                                                                             
src = pathlib.Path('train_local.py').read_text()                                                                                                                                                           
src = src.replace(                                                                                                                                                                                         
  'num_generations=CFG.num_generations,\n        max_completion_length=CFG.max_new_tokens,',                                                                                                             
  'num_generations=CFG.num_generations,\n        generation_batch_size=CFG.num_generations,\n        max_completion_length=CFG.max_new_tokens,',                                                         
)                                                                                                                                                                                                          
pathlib.Path('train_local.py').write_text(src)                                                                                                                                                             
print('Added generation_batch_size ✓')

Added generation_batch_size ✓


In [19]:
import os, sys, time                                                                                                                                                                                       
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'                                                                                                                                                        
sys.path.insert(0, '.')                                                                                                                                                                                    
                                                                                                                                                                                                         
t0 = time.time()                                                                                                                                                                                           
%run train_local.py                                                                                                                                                                                        
elapsed = (time.time() - t0) / 60                                                                                                                                                                          
print(f'\nTraining finished in {elapsed:.1f} min') 

[gpu] Tesla T4 | 14.6 GB
[model] Unsloth unavailable (ModuleNotFoundError: No module named 'unsloth'). Falling back to TRL + bitsandbytes.
[model] Loading Qwen/Qwen2.5-3B-Instruct via transformers (bf16, no quant)...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 7,372,800 || all params: 3,093,311,488 || trainable%: 0.2383
[model] Loaded with backend=trl
[data] Phase distribution: {'opening': 22, 'negotiation': 26, 'resolution': 10, 'other': 0, 'terminal': 6}
[data] Built dataset of 64 prompts


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


[train] Starting GRPO...


Step,Training Loss
2,0.115815
4,-0.109036
6,0.000000
8,0.000001
10,0.000001
12,0.000001
14,0.038383
16,-0.031091
18,0.119744
20,-0.166240


[train] Done in 11.8 min
[train] Adapter saved to ./crisis-negotiator-trained
[train] Log saved to crisis_training_log.json (256 entries)
[curriculum] {'current_tier': 'easy', 'history_lengths': {'easy': 58, 'medium': 0, 'hard': 0}, 'recent_avg': {'easy': 0.363, 'medium': 0.0, 'hard': 0.0}}
[plot] Saved reward_curve_training.png  (256 steps, last-10 mean=0.2980)

Training finished in 11.9 min


In [20]:
# Cell 7 — Plot training reward curve
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

with open('crisis_training_log.json') as f:
    log = json.load(f)

steps   = [e['global_step'] for e in log]
rewards = [e['reward']      for e in log]
phases  = [e.get('obs_phase', 'opening') for e in log]

window  = min(16, len(rewards))
rolling = np.convolve(rewards, np.ones(window) / window, mode='valid')
roll_x  = steps[window - 1:]

PHASE_COLORS = {'opening': '#4C72B0', 'negotiation': '#DD8452', 'resolution': '#55A868'}
colours = [PHASE_COLORS.get(p, '#999999') for p in phases]

fig, ax = plt.subplots(figsize=(11, 4))
ax.scatter(steps, rewards, c=colours, alpha=0.35, s=14, zorder=2)
ax.plot(roll_x, rolling, color='black', linewidth=2, label=f'rolling mean (w={window})', zorder=3)
ax.axhline(0.755, color='grey',    linestyle='--', linewidth=1, label='random baseline 0.755')
ax.axhline(0.950, color='#2ca02c', linestyle='--', linewidth=1, label='heuristic baseline 0.950')
ax.set_xlabel('GRPO Training Step')
ax.set_ylabel('Blended Episode Reward')
ax.set_title('Crisis Negotiator — GRPO Training Progress')

phase_patches = [mpatches.Patch(color=c, label=p) for p, c in PHASE_COLORS.items()]
line_handles, line_labels = ax.get_legend_handles_labels()
ax.legend(handles=line_handles + phase_patches,
          labels=line_labels + list(PHASE_COLORS.keys()),
          fontsize=7, loc='lower right', ncol=2)

plt.tight_layout()
plt.savefig('reward_curve_training.png', dpi=150)
plt.show()

print(f'Steps: {len(log)}')
print(f'Mean reward (all)    : {np.mean(rewards):.4f}')
print(f'Mean reward (last 10): {np.mean(rewards[-10:]):.4f}')
print(f'Improvement: {np.mean(rewards[-10:]) - np.mean(rewards[:10]):+.4f}')

Steps: 256
Mean reward (all)    : 0.3000
Mean reward (last 10): 0.2980
Improvement: -0.0510


In [25]:
import sys                                                                                                                                                                                                 
sys.argv = [                                                                                                                                                                                               
  'eval_baselines.py',                                                                                                                                                                                   
  '--n', '10',                                                                                                                                                                                           
  '--difficulties', 'easy,medium,hard',                                                                                                                                                                  
  '--include-trained',                                                                                                                                                                                   
  '--adapter-dir', './crisis-negotiator-trained',                                                                                                                                                        
  '--base-model', 'Qwen/Qwen2.5-3B-Instruct',                                                                                                                                                            
  '--skip-random',                                                                                                                                                                                       
  '--skip-heuristic',                                                                                                                                                                                    
]                                                                                                                                                                                                          
%run eval_baselines.py

=== Evaluating RANDOM policy ===
  [random] ep 1/30 (easy) -> reward=0.818 steps=14
  [random] ep 2/30 (medium) -> reward=0.854 steps=8
  [random] ep 3/30 (hard) -> reward=0.345 steps=18
  [random] ep 4/30 (easy) -> reward=0.789 steps=12
  [random] ep 5/30 (medium) -> reward=0.785 steps=14
  [random] ep 6/30 (hard) -> reward=0.800 steps=20
  [random] ep 7/30 (easy) -> reward=0.870 steps=8
  [random] ep 8/30 (medium) -> reward=0.822 steps=10
  [random] ep 9/30 (hard) -> reward=0.791 steps=22
  [random] ep 10/30 (easy) -> reward=0.607 steps=15
  [random] ep 11/30 (medium) -> reward=0.784 steps=12
  [random] ep 12/30 (hard) -> reward=0.798 steps=17
  [random] ep 13/30 (easy) -> reward=0.622 steps=12
  [random] ep 14/30 (medium) -> reward=0.879 steps=21
  [random] ep 15/30 (hard) -> reward=0.633 steps=22
  [random] ep 16/30 (easy) -> reward=0.838 steps=18
  [random] ep 17/30 (medium) -> reward=0.877 steps=7
  [random] ep 18/30 (hard) -> reward=0.797 steps=19
  [random] ep 19/30 (easy) -> r

In [27]:
from eval_baselines import TrainedPolicy, run_episodes, summarize                                                                                                                                          
import json                                                                                                                                                                                                
                                                                                                                                                                                                         
print("=== Evaluating TRAINED policy ===")                                                                                                                                                                 
trained = TrainedPolicy('./crisis-negotiator-trained', 'Qwen/Qwen2.5-3B-Instruct')                                                                                                                         
results = run_episodes(trained, 30, ['easy','medium','hard'], seed_offset=10_000)                                                                                                                          
summary = summarize(results)                                                                                                                                                                               
print(json.dumps(summary, indent=2))

=== Evaluating TRAINED policy ===


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

  [trained] ep 1/30 (easy) -> reward=0.944 steps=3
  [trained] ep 2/30 (medium) -> reward=0.967 steps=3
  [trained] ep 3/30 (hard) -> reward=0.929 steps=11
  [trained] ep 4/30 (easy) -> reward=0.957 steps=4
  [trained] ep 5/30 (medium) -> reward=0.923 steps=7
  [trained] ep 6/30 (hard) -> reward=0.931 steps=8
  [trained] ep 7/30 (easy) -> reward=0.945 steps=3
  [trained] ep 8/30 (medium) -> reward=0.970 steps=5
  [trained] ep 9/30 (hard) -> reward=0.962 steps=9
  [trained] ep 10/30 (easy) -> reward=0.971 steps=10
  [trained] ep 11/30 (medium) -> reward=0.931 steps=4
  [trained] ep 12/30 (hard) -> reward=0.990 steps=6
  [trained] ep 13/30 (easy) -> reward=0.935 steps=7
  [trained] ep 14/30 (medium) -> reward=0.931 steps=9
  [trained] ep 15/30 (hard) -> reward=0.972 steps=10
  [trained] ep 16/30 (easy) -> reward=0.959 steps=10
  [trained] ep 17/30 (medium) -> reward=0.937 steps=5
  [trained] ep 18/30 (hard) -> reward=0.946 steps=6
  [trained] ep 19/30 (easy) -> reward=0.922 steps=8
  [tr

In [28]:
import os, pathlib                                                                                                                                                                                         
adapter_dir = pathlib.Path('./crisis-negotiator-trained')                                                                                                                                                  
if adapter_dir.exists():                                                                                                                                                                                   
  print('Adapter files:', os.listdir(adapter_dir))                                                                                                                                                       
else:                                                                                                                                                                                                      
  print('Adapter directory not found!')

Adapter files: ['adapter_model.safetensors', 'ref', 'chat_template.jinja', 'tokenizer.json', 'adapter_config.json', 'README.md', 'checkpoint-240', 'tokenizer_config.json', 'checkpoint-256']


In [29]:
# Cell 9 — Save artifacts
import shutil, pathlib

artifacts = [
    'crisis-negotiator-trained',
    'crisis_training_log.json',
    'reward_curve_training.png',
    'eval_summary.json',
    'reward_curve.png',
]
for a in artifacts:
    p = pathlib.Path(a)
    if p.exists():
        print(f'  ✓ {a} ({p.stat().st_size / 1024:.0f} KB)' if p.is_file() else f'  ✓ {a}/ (dir)')
    else:
        print(f'  ✗ {a} — not found')

# Zip adapter for easy download
if pathlib.Path('crisis-negotiator-trained').exists():
    shutil.make_archive('crisis-negotiator-trained', 'zip', '.', 'crisis-negotiator-trained')
    print(f'\n📦 crisis-negotiator-trained.zip ready for download')

  ✓ crisis-negotiator-trained/ (dir)
  ✓ crisis_training_log.json (206 KB)
  ✓ reward_curve_training.png (121 KB)
  ✓ eval_summary.json (1 KB)
  ✓ reward_curve.png (63 KB)

📦 crisis-negotiator-trained.zip ready for download


In [30]:
import json, pathlib                                                                                                                                                                                       
pathlib.Path('eval_trained.json').write_text(json.dumps(results, indent=2))                                                                                                                                
                                                                                                                                                                                                         
# Update eval_summary with all 3 policies                                                                                                                                                                  
summary = {}                                                                                                                                                                                               
for name, path in [('random','eval_random.json'),('heuristic','eval_heuristic.json')]:                                                                                                                     
  if pathlib.Path(path).exists():                                                                                                                                                                        
      summary[name] = json.loads(pathlib.Path(path).read_text())                                                                                                                                         
summary['trained'] = results                                                                                                                                                                               
# Re-summarize                                                                                                                                                                                             
from eval_baselines import summarize                                                                                                                                                                       
final = {k: summarize(v) if isinstance(v, list) else v for k, v in summary.items()}                                                                                                                        
pathlib.Path('eval_summary.json').write_text(json.dumps(final, indent=2))                                                                                                                                  
print('Saved eval_trained.json + eval_summary.json')

Saved eval_trained.json + eval_summary.json
